In [5]:
import pandas as pd

room_booking_requests = pd.read_csv(
    "../data/raw/room_booking_requests.csv",
    parse_dates=[
        "requested_start_datetime",
        "requested_end_datetime"
    ]
)

employees = pd.read_csv("../data/raw/employees.csv")

print("Room booking requests:", room_booking_requests.shape)
print("Employees:", employees.shape)

Room booking requests: (40000, 10)
Employees: (1000, 10)


In [6]:
latest_date = room_booking_requests["booking_date"].max()

recent_start_date = (
    pd.to_datetime(latest_date)
    - pd.Timedelta(days=29)
)

recent_requests = room_booking_requests[
    pd.to_datetime(room_booking_requests["booking_date"])
    >= recent_start_date
].copy()

print("Latest booking date:", latest_date)
print("Recent window start:", recent_start_date.date())
print("Recent requests:", len(recent_requests))
print("Employees with recent activity:", recent_requests["employee_id"].nunique())

Latest booking date: 2025-12-31
Recent window start: 2025-12-02
Recent requests: 3266
Employees with recent activity: 880


In [7]:
#  ==========================================================================================================================================
#                                                           CHECKING RECENT BEHAVIOUR
# ===========================================================================================================================================

latest_date = room_booking_requests["booking_date"].max()

recent_start_date = (
    pd.to_datetime(latest_date)
    - pd.Timedelta(days=29)
)

recent_requests = room_booking_requests[
    pd.to_datetime(room_booking_requests["booking_date"])
    >= recent_start_date
].copy()

print("Latest booking date:", latest_date)
print("Recent window start:", recent_start_date.date())
print("Recent window end:", pd.to_datetime(latest_date).date())
print("Recent requests:", len(recent_requests))
print(
    "Employees with recent activity:",
    recent_requests["employee_id"].nunique()
)

Latest booking date: 2025-12-31
Recent window start: 2025-12-02
Recent window end: 2025-12-31
Recent requests: 3266
Employees with recent activity: 880


In [8]:
#  ==========================================================================================================================================
#                                                       1.    CHECKING RECENT BEHAVIOUR ( ROOM - 30-DAYS )
# ===========================================================================================================================================


recent_room_type_behavior = (
    recent_requests
    .groupby(["employee_id", "requested_room_type"])
    .size()
    .reset_index(name="request_count")
)

recent_room_type_behavior["percentage"] = (
    recent_room_type_behavior["request_count"]
    / recent_room_type_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

recent_room_type_behavior.head(20)

,employee_id,requested_room_type,request_count,percentage
0,E0001,Meeting,4,100.000000
1,E0002,Meeting,1,100.000000
2,E0003,Meeting,2,100.000000
3,E0004,Meeting,2,100.000000
4,E0005,Meeting,2,100.000000
5,E0006,Conference,4,100.000000
6,E0007,Conference,4,100.000000
7,E0008,Meeting,1,100.000000
8,E0009,Training,1,100.000000
9,E0010,Meeting,5,100.000000


In [9]:
recent_room_type_diversity = (
    recent_room_type_behavior
    .groupby("employee_id")
    .size()
    .value_counts()
    .sort_index()
)

print(recent_room_type_diversity)

1    565
2    250
3     55
4     10
Name: count, dtype: int64


In [10]:
#  ==========================================================================================================================================
#                                                      2.     CHECKING RECENT BEHAVIOUR ( FLOOR - 30-DAYS )
# ===========================================================================================================================================

recent_floor_behavior = (
    recent_requests
    .groupby(["employee_id", "requested_floor"])
    .size()
    .reset_index(name="request_count")
)

recent_floor_behavior["percentage"] = (
    recent_floor_behavior["request_count"]
    / recent_floor_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

recent_floor_behavior.head(20)

,employee_id,requested_floor,request_count,percentage
0,E0001,1,1,25.000000
1,E0001,2,3,75.000000
2,E0002,1,1,100.000000
3,E0003,5,2,100.000000
4,E0004,1,2,100.000000
5,E0005,5,2,100.000000
6,E0006,2,1,25.000000
7,E0006,3,2,50.000000
8,E0006,5,1,25.000000
9,E0007,4,4,100.000000


In [11]:
recent_floor_diversity = (
    recent_floor_behavior
    .groupby("employee_id")
    .size()
    .value_counts()
    .sort_index()
)

print(recent_floor_diversity)

1    566
2    244
3     62
4      8
Name: count, dtype: int64


In [12]:
#  ==========================================================================================================================================
#                                                     3.      CHECKING RECENT BEHAVIOUR ( CAPACITY - 30-DAYS )
# ===========================================================================================================================================


recent_capacity_behavior = (
    recent_requests
    .groupby(["employee_id", "requested_capacity"])
    .size()
    .reset_index(name="request_count")
)

recent_capacity_behavior["percentage"] = (
    recent_capacity_behavior["request_count"]
    / recent_capacity_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

recent_capacity_behavior.head(20)

,employee_id,requested_capacity,request_count,percentage
0,E0001,2,1,25.000000
1,E0001,4,2,50.000000
2,E0001,6,1,25.000000
3,E0002,4,1,100.000000
4,E0003,4,2,100.000000
5,E0004,6,2,100.000000
6,E0005,8,1,50.000000
7,E0005,10,1,50.000000
8,E0006,4,2,50.000000
9,E0006,6,1,25.000000


In [13]:
recent_capacity_diversity = (
    recent_capacity_behavior
    .groupby("employee_id")
    .size()
    .value_counts()
    .sort_index()
)

print(recent_capacity_diversity)

1    245
2    320
3    244
4     57
5     13
6      1
Name: count, dtype: int64


In [14]:
#  ==========================================================================================================================================
#                                                      4.     CHECKING RECENT BEHAVIOUR ( START HOUR - 30-DAYS )
# ===========================================================================================================================================


recent_start_hour_behavior = (
    recent_requests
    .assign(
        requested_start_hour=
        recent_requests["requested_start_datetime"].dt.hour
    )
    .groupby(["employee_id", "requested_start_hour"])
    .size()
    .reset_index(name="request_count")
)

recent_start_hour_behavior["percentage"] = (
    recent_start_hour_behavior["request_count"]
    / recent_start_hour_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

recent_start_hour_behavior.head(20)

,employee_id,requested_start_hour,request_count,percentage
0,E0001,8,2,50.000000
1,E0001,9,2,50.000000
2,E0002,9,1,100.000000
3,E0003,9,2,100.000000
4,E0004,8,1,50.000000
5,E0004,9,1,50.000000
6,E0005,9,1,50.000000
7,E0005,10,1,50.000000
8,E0006,9,1,25.000000
9,E0006,10,2,50.000000


In [15]:
recent_start_hour_diversity = (
    recent_start_hour_behavior
    .groupby("employee_id")
    .size()
    .value_counts()
    .sort_index()
)

print(recent_start_hour_diversity)

1    338
2    400
3    142
Name: count, dtype: int64


In [16]:
recent_flexibility_behavior = (
    recent_requests
    .groupby(["employee_id", "max_time_flexibility_minutes"])
    .size()
    .reset_index(name="request_count")
)

recent_flexibility_behavior["percentage"] = (
    recent_flexibility_behavior["request_count"]
    / recent_flexibility_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

recent_flexibility_behavior.head(20)

,employee_id,max_time_flexibility_minutes,request_count,percentage
0,E0001,0,1,25.0
1,E0001,15,1,25.0
2,E0001,30,1,25.0
3,E0001,120,1,25.0
4,E0002,30,1,100.0
5,E0003,0,1,50.0
6,E0003,30,1,50.0
7,E0004,30,2,100.0
8,E0005,15,1,50.0
9,E0005,120,1,50.0


In [17]:
room_type_behavior = (
    room_booking_requests
    .groupby(["employee_id", "requested_room_type"])
    .size()
    .reset_index(name="request_count")
)

room_type_behavior["percentage"] = (
    room_type_behavior["request_count"]
    / room_type_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

room_type_features = (
    room_type_behavior
    .pivot(
        index="employee_id",
        columns="requested_room_type",
        values="percentage"
    )
    .fillna(0)
    .reset_index()
)

room_type_features.head()

requested_room_type,employee_id,Conference,Focus,Meeting,Training
0,E0001,0.000000,7.692308,89.230769,3.076923
1,E0002,40.000000,0.000000,60.000000,0.000000
2,E0003,0.000000,0.000000,100.000000,0.000000
3,E0004,7.142857,3.571429,82.142857,7.142857
4,E0005,0.000000,0.000000,100.000000,0.000000


In [18]:
#  ==========================================================================================================================================
#                                                    PERCENTAGE SHIFT DEVIATION ( ROOMS )
# ===========================================================================================================================================


recent_room_type_features = (
    recent_room_type_behavior
    .pivot(
        index="employee_id",
        columns="requested_room_type",
        values="percentage"
    )
    .fillna(0)
    .reset_index()
)

room_type_columns = [
    "Conference",
    "Focus",
    "Meeting",
    "Training"
]

for col in room_type_columns:
    if col not in recent_room_type_features.columns:
        recent_room_type_features[col] = 0

room_type_comparison = (
    room_type_features
    .merge(
        recent_room_type_features[
            ["employee_id"] + room_type_columns
        ],
        on="employee_id",
        how="inner",
        suffixes=("_long_term", "_recent")
    )
)

room_type_comparison["room_type_shift_pct"] = (
    sum(
        (
            room_type_comparison[f"{col}_long_term"]
            - room_type_comparison[f"{col}_recent"]
        ).abs()
        for col in room_type_columns
    ) / 2
)

room_type_comparison[
    [
        "employee_id",
        "room_type_shift_pct"
    ]
].head(20)

requested_room_type,employee_id,room_type_shift_pct
0,E0001,10.769231
1,E0002,40.000000
2,E0003,0.000000
3,E0004,17.857143
4,E0005,0.000000
5,E0006,12.195122
6,E0007,9.375000
7,E0008,13.157895
8,E0009,0.000000
9,E0010,16.666667


In [19]:
room_type_comparison["room_type_shift_pct"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
)

count    880.000000
mean      16.798088
std       14.896692
min        0.000000
25%        7.142857
50%       13.245614
75%       21.555950
90%       34.294643
95%       44.444444
max       96.666667
Name: room_type_shift_pct, dtype: float64

In [20]:
floor_behavior = (
    room_booking_requests
    .groupby(["employee_id", "requested_floor"])
    .size()
    .reset_index(name="request_count")
)

floor_behavior["percentage"] = (
    floor_behavior["request_count"]
    / floor_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

floor_features = (
    floor_behavior
    .pivot(
        index="employee_id",
        columns="requested_floor",
        values="percentage"
    )
    .fillna(0)
    .reset_index()
)

floor_features.columns = [
    "employee_id",
    "floor_1_pct",
    "floor_2_pct",
    "floor_3_pct",
    "floor_4_pct",
    "floor_5_pct"
]

floor_features.head()

,employee_id,floor_1_pct,floor_2_pct,floor_3_pct,floor_4_pct,floor_5_pct
0,E0001,13.846154,75.384615,4.615385,1.538462,4.615385
1,E0002,100.000000,0.000000,0.000000,0.000000,0.000000
2,E0003,0.000000,0.000000,0.000000,0.000000,100.000000
3,E0004,92.857143,3.571429,3.571429,0.000000,0.000000
4,E0005,0.000000,0.000000,0.000000,0.000000,100.000000


In [21]:
recent_floor_behavior = (
    recent_requests
    .groupby(["employee_id", "requested_floor"])
    .size()
    .reset_index(name="request_count")
)

recent_floor_behavior["percentage"] = (
    recent_floor_behavior["request_count"]
    / recent_floor_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

recent_floor_behavior.head(10)

,employee_id,requested_floor,request_count,percentage
0,E0001,1,1,25.0
1,E0001,2,3,75.0
2,E0002,1,1,100.0
3,E0003,5,2,100.0
4,E0004,1,2,100.0
5,E0005,5,2,100.0
6,E0006,2,1,25.0
7,E0006,3,2,50.0
8,E0006,5,1,25.0
9,E0007,4,4,100.0


In [22]:
recent_floor_features = (
    recent_floor_behavior
    .pivot(
        index="employee_id",
        columns="requested_floor",
        values="percentage"
    )
    .fillna(0)
    .reset_index()
)

recent_floor_features.columns = [
    "employee_id",
    "floor_1_pct",
    "floor_2_pct",
    "floor_3_pct",
    "floor_4_pct",
    "floor_5_pct"
]

recent_floor_features.head()

,employee_id,floor_1_pct,floor_2_pct,floor_3_pct,floor_4_pct,floor_5_pct
0,E0001,25.0,75.0,0.0,0.0,0.0
1,E0002,100.0,0.0,0.0,0.0,0.0
2,E0003,0.0,0.0,0.0,0.0,100.0
3,E0004,100.0,0.0,0.0,0.0,0.0
4,E0005,0.0,0.0,0.0,0.0,100.0


In [23]:
floor_comparison = (
    floor_features
    .merge(
        recent_floor_features,
        on="employee_id",
        how="inner",
        suffixes=("_long_term", "_recent")
    )
)

floor_comparison.head()

,employee_id,floor_1_pct_long_term,floor_2_pct_long_term,floor_3_pct_long_term,floor_4_pct_long_term,floor_5_pct_long_term,floor_1_pct_recent,floor_2_pct_recent,floor_3_pct_recent,floor_4_pct_recent,floor_5_pct_recent
0,E0001,13.846154,75.384615,4.615385,1.538462,4.615385,25.0,75.0,0.0,0.0,0.0
1,E0002,100.000000,0.000000,0.000000,0.000000,0.000000,100.0,0.0,0.0,0.0,0.0
2,E0003,0.000000,0.000000,0.000000,0.000000,100.000000,0.0,0.0,0.0,0.0,100.0
3,E0004,92.857143,3.571429,3.571429,0.000000,0.000000,100.0,0.0,0.0,0.0,0.0
4,E0005,0.000000,0.000000,0.000000,0.000000,100.000000,0.0,0.0,0.0,0.0,100.0


In [24]:
#  ==========================================================================================================================================
#                                                    PERCENTAGE SHIFT DEVIATION ( FLOOR )
# ===========================================================================================================================================

floor_columns = [
    "floor_1",
    "floor_2",
    "floor_3",
    "floor_4",
    "floor_5"
]

floor_comparison["floor_shift_pct"] = (
    sum(
        (
            floor_comparison[f"{col}_pct_long_term"]
            - floor_comparison[f"{col}_pct_recent"]
        ).abs()
        for col in floor_columns
    ) / 2
)

floor_comparison[
    ["employee_id", "floor_shift_pct"]
].head(10)

,employee_id,floor_shift_pct
0,E0001,11.153846
1,E0002,0.000000
2,E0003,0.000000
3,E0004,7.142857
4,E0005,0.000000
5,E0006,37.804878
6,E0007,3.125000
7,E0008,0.000000
8,E0009,22.222222
9,E0010,30.000000


In [25]:
capacity_behavior = (
    room_booking_requests
    .groupby(["employee_id", "requested_capacity"])
    .size()
    .reset_index(name="request_count")
)

capacity_behavior["percentage"] = (
    capacity_behavior["request_count"]
    / capacity_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

capacity_behavior.head(10)

,employee_id,requested_capacity,request_count,percentage
0,E0001,2,14,21.538462
1,E0001,4,32,49.230769
2,E0001,6,14,21.538462
3,E0001,8,2,3.076923
4,E0001,10,2,3.076923
5,E0001,12,1,1.538462
6,E0002,2,2,20.000000
7,E0002,4,6,60.000000
8,E0002,6,1,10.000000
9,E0002,10,1,10.000000


In [26]:
capacity_features = (
    capacity_behavior
    .pivot(
        index="employee_id",
        columns="requested_capacity",
        values="percentage"
    )
    .fillna(0)
    .reset_index()
)

capacity_features.head()

requested_capacity,employee_id,2,4,6,8,10,12,16
0,E0001,21.538462,49.230769,21.538462,3.076923,3.076923,1.538462,0.000000
1,E0002,20.000000,60.000000,10.000000,0.000000,10.000000,0.000000,0.000000
2,E0003,0.000000,57.142857,42.857143,0.000000,0.000000,0.000000,0.000000
3,E0004,0.000000,14.285714,50.000000,25.000000,0.000000,7.142857,3.571429
4,E0005,12.500000,0.000000,0.000000,12.500000,62.500000,12.500000,0.000000


In [27]:
capacity_features.columns = [
    "employee_id",
    "capacity_2_pct",
    "capacity_4_pct",
    "capacity_6_pct",
    "capacity_8_pct",
    "capacity_10_pct",
    "capacity_12_pct",
    "capacity_16_pct"
]

capacity_features.head()

,employee_id,capacity_2_pct,capacity_4_pct,capacity_6_pct,capacity_8_pct,capacity_10_pct,capacity_12_pct,capacity_16_pct
0,E0001,21.538462,49.230769,21.538462,3.076923,3.076923,1.538462,0.000000
1,E0002,20.000000,60.000000,10.000000,0.000000,10.000000,0.000000,0.000000
2,E0003,0.000000,57.142857,42.857143,0.000000,0.000000,0.000000,0.000000
3,E0004,0.000000,14.285714,50.000000,25.000000,0.000000,7.142857,3.571429
4,E0005,12.500000,0.000000,0.000000,12.500000,62.500000,12.500000,0.000000


In [28]:
recent_capacity_behavior = (
    recent_requests
    .groupby(["employee_id", "requested_capacity"])
    .size()
    .reset_index(name="request_count")
)

recent_capacity_behavior["percentage"] = (
    recent_capacity_behavior["request_count"]
    / recent_capacity_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

recent_capacity_behavior.head(10)

,employee_id,requested_capacity,request_count,percentage
0,E0001,2,1,25.0
1,E0001,4,2,50.0
2,E0001,6,1,25.0
3,E0002,4,1,100.0
4,E0003,4,2,100.0
5,E0004,6,2,100.0
6,E0005,8,1,50.0
7,E0005,10,1,50.0
8,E0006,4,2,50.0
9,E0006,6,1,25.0


In [29]:
recent_capacity_features = (
    recent_capacity_behavior
    .pivot(
        index="employee_id",
        columns="requested_capacity",
        values="percentage"
    )
    .fillna(0)
    .reset_index()
)

recent_capacity_features.columns = [
    "employee_id",
    "capacity_2_pct",
    "capacity_4_pct",
    "capacity_6_pct",
    "capacity_8_pct",
    "capacity_10_pct",
    "capacity_12_pct",
    "capacity_16_pct"
]

recent_capacity_features.head()

,employee_id,capacity_2_pct,capacity_4_pct,capacity_6_pct,capacity_8_pct,capacity_10_pct,capacity_12_pct,capacity_16_pct
0,E0001,25.0,50.0,25.0,0.0,0.0,0.0,0.0
1,E0002,0.0,100.0,0.0,0.0,0.0,0.0,0.0
2,E0003,0.0,100.0,0.0,0.0,0.0,0.0,0.0
3,E0004,0.0,0.0,100.0,0.0,0.0,0.0,0.0
4,E0005,0.0,0.0,0.0,50.0,50.0,0.0,0.0


In [30]:
capacity_comparison = (
    capacity_features
    .merge(
        recent_capacity_features,
        on="employee_id",
        how="inner",
        suffixes=("_long_term", "_recent")
    )
)

capacity_comparison.head()

,employee_id,capacity_2_pct_long_term,capacity_4_pct_long_term,capacity_6_pct_long_term,capacity_8_pct_long_term,capacity_10_pct_long_term,capacity_12_pct_long_term,capacity_16_pct_long_term,capacity_2_pct_recent,capacity_4_pct_recent,capacity_6_pct_recent,capacity_8_pct_recent,capacity_10_pct_recent,capacity_12_pct_recent,capacity_16_pct_recent
0,E0001,21.538462,49.230769,21.538462,3.076923,3.076923,1.538462,0.000000,25.0,50.0,25.0,0.0,0.0,0.0,0.0
1,E0002,20.000000,60.000000,10.000000,0.000000,10.000000,0.000000,0.000000,0.0,100.0,0.0,0.0,0.0,0.0,0.0
2,E0003,0.000000,57.142857,42.857143,0.000000,0.000000,0.000000,0.000000,0.0,100.0,0.0,0.0,0.0,0.0,0.0
3,E0004,0.000000,14.285714,50.000000,25.000000,0.000000,7.142857,3.571429,0.0,0.0,100.0,0.0,0.0,0.0,0.0
4,E0005,12.500000,0.000000,0.000000,12.500000,62.500000,12.500000,0.000000,0.0,0.0,0.0,50.0,50.0,0.0,0.0


In [31]:
#  ==========================================================================================================================================
#                                                    PERCENTAGE SHIFT DEVIATION (CAPACITY )
# ===========================================================================================================================================

capacity_columns = [
    "capacity_2",
    "capacity_4",
    "capacity_6",
    "capacity_8",
    "capacity_10",
    "capacity_12",
    "capacity_16"
]

capacity_comparison["capacity_shift_pct"] = (
    sum(
        (
            capacity_comparison[f"{col}_pct_long_term"]
            - capacity_comparison[f"{col}_pct_recent"]
        ).abs()
        for col in capacity_columns
    ) / 2
)

capacity_comparison[
    ["employee_id", "capacity_shift_pct"]
].head(10)

,employee_id,capacity_shift_pct
0,E0001,7.692308
1,E0002,40.000000
2,E0003,42.857143
3,E0004,50.000000
4,E0005,37.500000
5,E0006,33.536585
6,E0007,25.000000
7,E0008,94.736842
8,E0009,66.666667
9,E0010,26.666667


In [32]:
start_hour_behavior = (
    room_booking_requests
    .assign(
        requested_start_hour=
        room_booking_requests["requested_start_datetime"].dt.hour
    )
    .groupby(["employee_id", "requested_start_hour"])
    .size()
    .reset_index(name="request_count")
)

start_hour_behavior["percentage"] = (
    start_hour_behavior["request_count"]
    / start_hour_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

start_hour_behavior.head(10)

,employee_id,requested_start_hour,request_count,percentage
0,E0001,8,9,13.846154
1,E0001,9,47,72.307692
2,E0001,10,9,13.846154
3,E0002,9,8,80.000000
4,E0002,10,2,20.000000
5,E0003,8,2,28.571429
6,E0003,9,5,71.428571
7,E0004,8,3,10.714286
8,E0004,9,20,71.428571
9,E0004,10,5,17.857143


In [33]:
start_hour_features = (
    start_hour_behavior
    .pivot(
        index="employee_id",
        columns="requested_start_hour",
        values="percentage"
    )
    .fillna(0)
    .reset_index()
)

start_hour_features.head()

requested_start_hour,employee_id,8,9,10,11
0,E0001,13.846154,72.307692,13.846154,0.0
1,E0002,0.000000,80.000000,20.000000,0.0
2,E0003,28.571429,71.428571,0.000000,0.0
3,E0004,10.714286,71.428571,17.857143,0.0
4,E0005,12.500000,25.000000,62.500000,0.0


In [34]:
start_hour_features.columns = [
    "employee_id",
    "start_hour_8_pct",
    "start_hour_9_pct",
    "start_hour_10_pct",
    "start_hour_11_pct"
]

start_hour_features.head()

,employee_id,start_hour_8_pct,start_hour_9_pct,start_hour_10_pct,start_hour_11_pct
0,E0001,13.846154,72.307692,13.846154,0.0
1,E0002,0.000000,80.000000,20.000000,0.0
2,E0003,28.571429,71.428571,0.000000,0.0
3,E0004,10.714286,71.428571,17.857143,0.0
4,E0005,12.500000,25.000000,62.500000,0.0


In [35]:
recent_start_hour_behavior = (
    recent_requests
    .assign(
        requested_start_hour=
        recent_requests["requested_start_datetime"].dt.hour
    )
    .groupby(["employee_id", "requested_start_hour"])
    .size()
    .reset_index(name="request_count")
)

recent_start_hour_behavior["percentage"] = (
    recent_start_hour_behavior["request_count"]
    / recent_start_hour_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

recent_start_hour_behavior.head(10)

,employee_id,requested_start_hour,request_count,percentage
0,E0001,8,2,50.0
1,E0001,9,2,50.0
2,E0002,9,1,100.0
3,E0003,9,2,100.0
4,E0004,8,1,50.0
5,E0004,9,1,50.0
6,E0005,9,1,50.0
7,E0005,10,1,50.0
8,E0006,9,1,25.0
9,E0006,10,2,50.0


In [36]:
recent_start_hour_features = (
    recent_start_hour_behavior
    .pivot(
        index="employee_id",
        columns="requested_start_hour",
        values="percentage"
    )
    .fillna(0)
    .reset_index()
)

recent_start_hour_features.head()

requested_start_hour,employee_id,8,9,10,11
0,E0001,50.0,50.0,0.0,0.0
1,E0002,0.0,100.0,0.0,0.0
2,E0003,0.0,100.0,0.0,0.0
3,E0004,50.0,50.0,0.0,0.0
4,E0005,0.0,50.0,50.0,0.0


In [37]:
recent_start_hour_features.columns = [
    "employee_id",
    "start_hour_8_pct",
    "start_hour_9_pct",
    "start_hour_10_pct",
    "start_hour_11_pct"
]

recent_start_hour_features.head()

,employee_id,start_hour_8_pct,start_hour_9_pct,start_hour_10_pct,start_hour_11_pct
0,E0001,50.0,50.0,0.0,0.0
1,E0002,0.0,100.0,0.0,0.0
2,E0003,0.0,100.0,0.0,0.0
3,E0004,50.0,50.0,0.0,0.0
4,E0005,0.0,50.0,50.0,0.0


In [38]:
start_hour_comparison = (
    start_hour_features
    .merge(
        recent_start_hour_features,
        on="employee_id",
        how="inner",
        suffixes=("_long_term", "_recent")
    )
)

start_hour_comparison.head()

,employee_id,start_hour_8_pct_long_term,start_hour_9_pct_long_term,start_hour_10_pct_long_term,start_hour_11_pct_long_term,start_hour_8_pct_recent,start_hour_9_pct_recent,start_hour_10_pct_recent,start_hour_11_pct_recent
0,E0001,13.846154,72.307692,13.846154,0.0,50.0,50.0,0.0,0.0
1,E0002,0.000000,80.000000,20.000000,0.0,0.0,100.0,0.0,0.0
2,E0003,28.571429,71.428571,0.000000,0.0,0.0,100.0,0.0,0.0
3,E0004,10.714286,71.428571,17.857143,0.0,50.0,50.0,0.0,0.0
4,E0005,12.500000,25.000000,62.500000,0.0,0.0,50.0,50.0,0.0


In [39]:
#  ==========================================================================================================================================
#                                                    PERCENTAGE SHIFT DEVIATION (  START HOUR  )
# ===========================================================================================================================================

start_hour_columns = [
    "start_hour_8",
    "start_hour_9",
    "start_hour_10",
    "start_hour_11"
]

start_hour_comparison["start_hour_shift_pct"] = (
    sum(
        (
            start_hour_comparison[f"{col}_pct_long_term"]
            - start_hour_comparison[f"{col}_pct_recent"]
        ).abs()
        for col in start_hour_columns
    ) / 2
)

start_hour_comparison[
    ["employee_id", "start_hour_shift_pct"]
].head(10)

,employee_id,start_hour_shift_pct
0,E0001,36.153846
1,E0002,20.000000
2,E0003,28.571429
3,E0004,39.285714
4,E0005,25.000000
5,E0006,23.170732
6,E0007,12.500000
7,E0008,18.421053
8,E0009,22.222222
9,E0010,6.666667


In [40]:
# ============================================================
# SAVE FINAL BEHAVIOUR FEATURES FOR ML NOTEBOOK
# ============================================================

room_type_comparison.to_csv(
    "../data/processed/room_type_behavior_features.csv",
    index=False
)

floor_comparison.to_csv(
    "../data/processed/floor_behavior_features.csv",
    index=False
)

capacity_comparison.to_csv(
    "../data/processed/capacity_behavior_features.csv",
    index=False
)

start_hour_comparison.to_csv(
    "../data/processed/start_hour_behavior_features.csv",
    index=False
)

print("Final behaviour feature tables saved successfully.")

Final behaviour feature tables saved successfully.
